# Correção e sinalização de qualidade do dataset de filmes

## tl;dr

Este notebook cria uma versão adicional do dataset tratado, sem modificar o notebook de EDA nem substituir `data/clean/dataset.csv`. Valores que podem ser recuperados com segurança recebem colunas validadas; problemas sem correção confiável permanecem documentados por flags.

## Contexto e métodos

A unidade de análise é um filme. O campo `ranking` é usado para conferir o alinhamento entre os arquivos bruto e tratado. Nenhum valor externo é inventado. Datas presentes por engano em `orcamento_texto` são recuperadas somente quando o ano extraído coincide com `ano`. Campos aparentemente copiados da linha anterior são anulados nas versões validadas.

### Premissas principais

- Diferenças de até US$ 1.000 entre receita mundial e a soma dos componentes são tratadas como tolerância operacional.
- Receita doméstica igual à internacional é considerada suspeita quando também há inconsistência na soma mundial.
- As colunas originais são preservadas para rastreabilidade; as correções ficam em novas colunas com sufixo `_validado` ou `_validada`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path("..").resolve() if Path("../data").exists() else Path(".").resolve()
ARQUIVO_RAW = RAIZ / "data" / "raw" / "Highest Holywood Grossing Movies.csv"
ARQUIVO_CLEAN = RAIZ / "data" / "clean" / "dataset.csv"
ARQUIVO_SAIDA = RAIZ / "data" / "clean" / "dataset_corrigido.csv"

for arquivo in [ARQUIVO_RAW, ARQUIVO_CLEAN]:
    if not arquivo.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {arquivo}")

## Dados

### 1. Carregar e reconciliar as fontes

In [ ]:
raw = pd.read_csv(ARQUIVO_RAW, keep_default_na=False)
clean_texto = pd.read_csv(ARQUIVO_CLEAN, dtype=str, keep_default_na=False)
clean = pd.read_csv(ARQUIVO_CLEAN)

if len(raw) != len(clean):
    raise ValueError(f"Contagem de linhas divergente: raw={len(raw)}, clean={len(clean)}")

ranking_raw = pd.to_numeric(raw["Unnamed: 0"], errors="raise").astype(int)
ranking_clean = pd.to_numeric(clean["ranking"], errors="raise").astype(int)
if not ranking_raw.equals(ranking_clean):
    raise ValueError("O alinhamento por ranking entre raw e clean não foi preservado.")

print(f"Raw: {raw.shape[0]:,} linhas x {raw.shape[1]} colunas")
print(f"Clean atual: {clean.shape[0]:,} linhas x {clean.shape[1]} colunas")

## Resultados

### 2. Detectar problemas de qualidade

In [ ]:
flags = pd.DataFrame(index=clean.index)
flags["flag_orcamento_invalido"] = clean["orcamento"].isna()
flags["flag_abertura_invalida"] = clean["abertura_domestica"].isna()

campos_suspeitos = [
    "data_lancamento_texto",
    "generos_texto",
    "duracao_texto",
    "classificacao",
]
iguais_ao_anterior = pd.Series(True, index=clean.index)
for coluna in campos_suspeitos:
    atual = clean[coluna].astype("string")
    iguais_ao_anterior &= atual.eq(atual.shift()).fillna(False)
flags["flag_campos_copiados_anterior"] = (
    flags["flag_orcamento_invalido"] & iguais_ao_anterior
)

data_atual = pd.to_datetime(clean["data_lancamento"], errors="coerce")
flags["flag_data_ano_inconsistente"] = (
    data_atual.isna() | data_atual.dt.year.ne(clean["ano"])
)

delta_receitas = (
    clean["bilheteria_mundial"]
    - clean["bilheteria_domestica"]
    - clean["bilheteria_internacional"]
)
flags["flag_receitas_inconsistentes"] = delta_receitas.abs().gt(1_000)
flags["flag_receitas_dom_int_iguais"] = (
    clean["bilheteria_domestica"].eq(clean["bilheteria_internacional"])
    & clean["bilheteria_domestica"].gt(0)
)


### 3. Criar colunas validadas e flags consolidadas

A correção é conservadora: mantém os dados originais e cria versões validadas. Quando não há evidência suficiente para corrigir um valor, a versão validada recebe ausência em vez de copiar um valor potencialmente incorreto.

In [ ]:
corrigido = clean_texto.copy()

texto_data_no_orcamento = clean["orcamento_texto"].astype("string").str.extract(
    r"^([A-Za-z]+ \d{1,2} \d{4})", expand=False
)
data_extraida = pd.to_datetime(
    texto_data_no_orcamento, format="%B %d %Y", errors="coerce"
)
flags["flag_data_recuperada_de_orcamento_texto"] = (
    flags["flag_campos_copiados_anterior"]
    & data_extraida.notna()
    & data_extraida.dt.year.eq(clean["ano"])
)

data_validada = data_atual.copy()
data_validada.loc[
    flags["flag_campos_copiados_anterior"] | flags["flag_data_ano_inconsistente"]
] = pd.NaT
data_validada.loc[flags["flag_data_recuperada_de_orcamento_texto"]] = data_extraida
corrigido["data_lancamento_validada"] = data_validada.dt.strftime("%Y-%m-%d")

corrigido["genero_principal_validado"] = clean["genero_principal"].astype("string")
corrigido.loc[flags["flag_campos_copiados_anterior"], "genero_principal_validado"] = pd.NA

corrigido["duracao_min_validada"] = clean["duracao_min"].astype("Float64")
corrigido.loc[flags["flag_campos_copiados_anterior"], "duracao_min_validada"] = pd.NA

corrigido["classificacao_validada"] = clean["classificacao"].astype("string")
corrigido.loc[flags["flag_campos_copiados_anterior"], "classificacao_validada"] = pd.NA

componentes_fortemente_suspeitos = (
    flags["flag_receitas_dom_int_iguais"]
    & flags["flag_receitas_inconsistentes"]
)
corrigido["bilheteria_domestica_validada"] = clean["bilheteria_domestica"].astype("Float64")
corrigido["bilheteria_internacional_validada"] = clean["bilheteria_internacional"].astype("Float64")
corrigido.loc[componentes_fortemente_suspeitos, [
    "bilheteria_domestica_validada",
    "bilheteria_internacional_validada",
]] = pd.NA
corrigido["bilheteria_mundial_validada"] = clean["bilheteria_mundial"].astype("Float64")

for coluna in flags.columns:
    corrigido[coluna] = flags[coluna].astype(bool)

flags_de_risco = [
    "flag_orcamento_invalido",
    "flag_abertura_invalida",
    "flag_campos_copiados_anterior",
    "flag_data_ano_inconsistente",
    "flag_receitas_inconsistentes",
    "flag_receitas_dom_int_iguais",
]
corrigido["registro_suspeito"] = corrigido[flags_de_risco].any(axis=1)
corrigido["qualidade_status"] = np.where(corrigido["registro_suspeito"], "Revisar", "OK")

rotulos = {
    "flag_orcamento_invalido": "orçamento inválido",
    "flag_abertura_invalida": "abertura inválida",
    "flag_campos_copiados_anterior": "campos copiados da linha anterior",
    "flag_data_ano_inconsistente": "data incompatível com o ano",
    "flag_receitas_inconsistentes": "componentes de receita não reconciliam",
    "flag_receitas_dom_int_iguais": "receitas doméstica e internacional idênticas",
}
corrigido["motivos_qualidade"] = [
    "; ".join(rotulos[coluna] for coluna in flags_de_risco if bool(linha[coluna]))
    for _, linha in corrigido[flags_de_risco].iterrows()
]

### 4. Resumir, validar e exportar

In [ ]:
resumo_flags = pd.DataFrame({
    "registros": corrigido[flags.columns].sum().astype(int),
    "percentual": corrigido[flags.columns].mean().mul(100),
}).sort_values("registros", ascending=False)
resumo_flags

In [ ]:
colunas_originais = clean.columns.tolist()
assert len(corrigido) == len(clean) == len(raw)
assert corrigido["ranking"].is_unique
assert corrigido[colunas_originais].equals(clean_texto[colunas_originais])
assert not corrigido.loc[
    corrigido["flag_data_recuperada_de_orcamento_texto"],
    "data_lancamento_validada",
].isna().any()
assert corrigido.loc[
    componentes_fortemente_suspeitos,
    ["bilheteria_domestica_validada", "bilheteria_internacional_validada"],
].isna().all().all()

ARQUIVO_SAIDA.parent.mkdir(parents=True, exist_ok=True)
corrigido.to_csv(ARQUIVO_SAIDA, index=False, encoding="utf-8")
recarregado = pd.read_csv(ARQUIVO_SAIDA)
assert recarregado.shape == corrigido.shape
assert recarregado["ranking"].astype(str).tolist() == corrigido["ranking"].astype(str).tolist()

print(f"Arquivo salvo em: {ARQUIVO_SAIDA}")
print(f"Dimensões: {corrigido.shape[0]:,} linhas x {corrigido.shape[1]} colunas")
print(f"Registros para revisão: {corrigido['registro_suspeito'].sum():,}")
print(f"Datas recuperadas com segurança: {corrigido['flag_data_recuperada_de_orcamento_texto'].sum():,}")

## Conclusões

- As colunas do dataset tratado original permanecem intactas na nova versão.
- As colunas validadas evitam reutilizar valores identificados como cópias da linha anterior.
- A bilheteria mundial é preservada; componentes fortemente suspeitos recebem ausência nas versões validadas.
- `registro_suspeito`, `qualidade_status` e `motivos_qualidade` permitem filtrar registros seguros ou revisar casos problemáticos.
- A correção é conservadora e não substitui uma nova coleta para gênero, duração, classificação, orçamento e componentes de receita ausentes ou contaminados.